# P09 — Diálogo, respuestas y umbral de confianza
## (a) Paso del protocolo y objetivo
**Paso P09**: definir cómo responde el chatbot: una respuesta fija `utter_<intención>` por cada una de las 54 intenciones (verificada contra el TUPA: compuerta **G4**), una regla para `nlu_fallback`, y el **umbral de confianza** con el que el asistente prefiere decir «no entendí» a responder mal. Sustenta el **OE2** (arquitectura) y es lo que usa el asistente local del pre-piloto.
**Origen:** textos del dominio y reglas (**Sintético**: código, no datos de personas); selección del umbral con datos **Reales** (cifras agregadas).

## (b) Código del repositorio que lo implementa
`aplicar_tupa.py` y `aplicar_respuestas.py` (aplican a `domain.yml` / `domain_v3.yml` las respuestas verificadas, con respaldo y *dry-run*), `humo_respuestas.py` (cada intención tiene su `utter_` sin placeholders), `fallback_threshold.py` (umbral en validación real y aplicación al test una vez) y `umbral_lote2.py` (umbral del lote 2, congelado antes de abrirlo). Entradas: `docs/tupa/Verificacion_TUPA_v10_4.xlsx`, `domain_v3.yml`. Salidas: `domain_v3.yml`, `logs/v3_real/umbral*.json`.

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("09"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,aplicar_respuestas.py,"P09, G4",Textos de respuesta (bloques A–E del 2026-10-08): TUPA v10_2 + textos del tesista (docs/tupa/respuestas_manual_20261008.yml) + nombres oficiales.,docs/tupa/respuestas_manual_20261008.yml + Verificacion_TUPA,domain.yml / domain_v3.yml actualizados; informe,python scripts/aplicar_respuestas.py --help,No,Documentado (con respaldo y dry-run)
1,aplicar_tupa.py,"P09, G4","Aplica a domain.yml las respuestas corregidas de la hoja de verificación del TUPA (docs/tupa/Verificacion_TUPA_v*.xlsx, la versión más alta).",docs/tupa/Verificacion_TUPA_v*.xlsx + domain,domain.yml / domain_v3.yml con respuestas verificadas; informe .md,python scripts/aplicar_tupa.py # dry-run,No,Documentado (con respaldo y dry-run)
2,fallback_threshold.py,T05 (umbral),Umbral de confianza (FallbackClassifier) — elección en validación real y evaluación única en test (protocolo V1.2).,Predicciones de validación,Umbral t de confianza elegido en validación,python scripts/fallback_threshold.py,No,Documentado
3,humo_respuestas.py,"P09, G4","Prueba de humo de respuestas: cada una de las 54 intenciones tiene su utter_<intención> con texto, sin placeholders ni YAML roto, y las respuestas coinciden entre domain.yml y domain_v3.yml (salvo el fallback).","domain.yml, domain_v3.yml",Prueba de humo: 54 intenciones con utter_<intención> sin placeholders,python scripts/humo_respuestas.py --help,No,Lógica recalculada (etapa 9)
4,umbral_lote2.py,Lote 2 (G3),Lote 2 — umbral de confianza (FallbackClassifier) elegido con datos del lote 1 y congelado ANTES de abrir el lote 2.,Validación cruzada por participante del lote 1,"logs/v3_real/lote2_umbral_congelado.json (t = 0,50)",python scripts/umbral_lote2.py --pred logs/v3_real/cv_participantes_ciclo1/predicciones_rasa.csv,No,Documentado


## (c) Celdas de código
### 1. Reglas de diálogo y mensaje de «no entendí» (Sintético)

In [3]:
dom = yaml.safe_load((BASE / "domain_v3.yml").read_text(encoding="utf-8"))
reglas = yaml.safe_load((BASE / "data/v3/rules_v3.yml").read_text(encoding="utf-8"))
rotulo(ORIGEN_SINTETICO, "dominio y reglas del chatbot")
print("Respuestas (utter_*) en domain_v3.yml:", len(dom["responses"]), "| intenciones:", len(dom["intents"]))
regla_fb = [r for r in reglas["rules"] if any(s.get("intent") == "nlu_fallback" for s in r.get("steps", []))]
print("Reglas para nlu_fallback:", [r["rule"] for r in regla_fb])
print("Texto de la respuesta «no entendí» (utter_no_entendi):", dom["responses"]["utter_no_entendi"][0]["text"])

▌ORIGEN: Sintético — dominio y reglas del chatbot
Respuestas (utter_*) en domain_v3.yml: 55 | intenciones: 55
Reglas para nlu_fallback: ['Responder nlu_fallback']
Texto de la respuesta «no entendí» (utter_no_entendi): No estoy seguro de haber entendido tu consulta. ¿Puedes reformularla o decirme qué trámite necesitas?


### 2. La regla de abstención con confianzas INVENTADAS (Simulado)
Mismo criterio del asistente local (`asistente_local.py`): **abstiene** si la confianza de la intención más probable es `< t` **o** si la diferencia con la segunda es `< 0,1` (ambigüedad). Los números de esta tabla son inventados para ilustrar la regla.

In [4]:
def decide(c1, c2, t=0.5, amb=0.1):
    return "abstiene (no entendí)" if (c1 < t or (c1 - c2) < amb) else "responde utter_<intención>"


casos = [(0.95, 0.01), (0.62, 0.10), (0.48, 0.20), (0.55, 0.50), (0.80, 0.75), (0.50, 0.30)]
rotulo(ORIGEN_SIMULADO, "confianzas inventadas — no son predicciones del modelo")
display(pd.DataFrame([{"confianza 1.ª": c1, "confianza 2.ª": c2, "diferencia": round(c1 - c2, 2), "decisión con t = 0,50": decide(c1, c2)} for c1, c2 in casos]))

▌ORIGEN: Simulado (demostración) — confianzas inventadas — no son predicciones del modelo


,confianza 1.ª,confianza 2.ª,diferencia,"decisión con t = 0,50"
0,0.95,0.01,0.94,responde utter_<intención>
1,0.62,0.10,0.52,responde utter_<intención>
2,0.48,0.20,0.28,abstiene (no entendí)
3,0.55,0.50,0.05,abstiene (no entendí)
4,0.80,0.75,0.05,abstiene (no entendí)
5,0.50,0.30,0.20,responde utter_<intención>


### 3. Cómo se eligió el umbral (Real, cifras guardadas)
- **Lote 1:** se eligió `t = 0,60` en la **validación real** (71 frases) maximizando *aciertos respondidos − 2 × errores respondidos*; sin leer el test.
- **Lote 2:** se eligió `t = 0,50` con la **validación cruzada por participante** de las 185 frases reales activas del lote 1 y se **congeló antes de abrir** el lote 2.

In [5]:
u1 = leer_json("logs/v3_real/umbral_congelado.json")
u2 = leer_json("logs/v3_real/lote2_umbral_congelado.json")
rotulo(ORIGEN_REAL, "umbrales congelados (cifras de selección; sin frases)")
display(pd.DataFrame([
    {"lote": "1 (validación real, 71 frases)", "t": u1["t"], "ambigüedad": u1["ambiguity_threshold"], "puntaje con umbral": u1["puntaje_validacion"], "puntaje sin umbral": u1["puntaje_sin_umbral_validacion"], "cobertura": "63,4 %", "precisión de lo respondido": "84,4 %"},
    {"lote": "2 (CV por participante, 185 frases)", "t": u2["t"], "ambigüedad": u2["ambiguity_threshold"], "puntaje con umbral": u2["puntaje_cv"], "puntaje sin umbral": u2["puntaje_sin_umbral_cv"], "cobertura": f"{u2['cobertura_cv']:.1%}", "precisión de lo respondido": f"{u2['precision_respondida_cv']:.1%}"}]))
print("El umbral MEJORA el puntaje en ambos casos (24 > 8 en el lote 1; 109 > 71 en el lote 2).")
txt = (BASE / "logs/v3_real/umbral_reporte.txt").read_text(encoding="utf-8").splitlines()
print("\nTabla de umbrales del lote 1 (logs/v3_real/umbral_reporte.txt):")
print("\n".join(l[:150] for l in txt[3:12]))
comparar("P09", "umbral t congelado para el modelo del lote 2", u2["t"], 0.5, "Protocolo V1.8", tol=0)
comparar("P09", "ambigüedad congelada", u2["ambiguity_threshold"], 0.1, "Protocolo V1.8", tol=0)

▌ORIGEN: Real — umbrales congelados (cifras de selección; sin frases)


,lote,t,ambigüedad,puntaje con umbral,puntaje sin umbral,cobertura,precisión de lo respondido
0,"1 (validación real, 71 frases)",0.6,0.1,24.0,8.0,"63,4 %","84,4 %"
1,"2 (CV por participante, 185 frases)",0.5,0.1,109.0,71.0,81.6%,90.7%


El umbral MEJORA el puntaje en ambos casos (24 > 8 en el lote 1; 109 > 71 en el lote 2).

Tabla de umbrales del lote 1 (logs/v3_real/umbral_reporte.txt):

         t  respondidas  aciertos_respondidos  errores_respondidos  abstenciones  correctas_perdidas  errores_atrapados cobertura precision_respondida
sin umbral           71                    50                   21             0                   0                  0    100.0%                70.4%
      0.30           61                    48                   13            10                   2                  8     85.9%                78.7%
      0.40           55                    44                   11            16                   6                 10     77.5%                80.0%
      0.50           50                    41                    9            21                   9                 12     70.4%                82.0%
      0.60           45                    38                    7            26          

### 4. Resultado del umbral en el test del lote 2 (Real, ya medido)
El asistente responde 248 de 267 frases (cobertura 92,9 %) y acierta 237 (precisión 95,6 %). Con el paquete privado se recalcula desde las predicciones guardadas; si no, se muestra el valor guardado.

In [6]:
if HAY_PRIVADO:
    pred = pd.read_csv(PRIV / "predicciones_lote2_sin_texto.csv", keep_default_na=False, encoding="utf-8")
    n = len(pred)
    abst = int((pred["predicted"] == "nlu_fallback").sum())
    aciertos = int((pred["intent"] == pred["predicted"]).sum())
    rotulo(ORIGEN_REAL, "recalculado desde predicciones guardadas (sin texto)")
    print(f"Frases: {n} · abstenciones (nlu_fallback): {abst} · respondidas: {n - abst} · aciertos: {aciertos}")
    cob, prec = (n - abst) / n, aciertos / (n - abst)
    print(f"Cobertura = {cob:.1%} · precisión de lo respondido = {prec:.1%}")
else:
    requiere_privado("predicciones del lote 2")
    rotulo(ORIGEN_REAL, "valores GUARDADOS (Protocolo V1.8 / Nota v11)")
    n, abst, aciertos = 267, 19, 237
    cob, prec = (n - abst) / n, aciertos / (n - abst)
    print(f"Frases: {n} · abstenciones: {abst} · aciertos: {aciertos} · cobertura {cob:.1%} · precisión de lo respondido {prec:.1%}")
comparar("P09", "abstenciones", abst, 19, "Protocolo V1.8 / Nota v11", tol=0)
comparar("P09", "cobertura", cob, 0.929, "Protocolo V1.8 (92,9 %)", tol=5e-4)
comparar("P09", "precisión de lo respondido", prec, 0.956, "Protocolo V1.8 (95,6 %)", tol=5e-4)
DECLARADAS.append("eval_lote2_resumen.json: la sección «umbral» (cobertura 98,9 %, precisión 89,8 %) es un artefacto declarado del script; las cifras oficiales son 92,9 % y 95,6 %.")

▌ORIGEN: Real — recalculado desde predicciones guardadas (sin texto)
Frases: 267 · abstenciones (nlu_fallback): 19 · respondidas: 248 · aciertos: 237
Cobertura = 92.9% · precisión de lo respondido = 95.6%


### 5. Respuestas verificadas contra el TUPA (G4)

In [7]:
dom = yaml.safe_load((BASE / "domain_v3.yml").read_text(encoding="utf-8"))
resp = dom["responses"]
con_marca = [k for k, v in resp.items() if "[Verificar" in json.dumps(v, ensure_ascii=False)]
sin_texto = [k for k, v in resp.items() if not v or not str(v[0].get("text", "")).strip()]
intents = [i if isinstance(i, str) else list(i)[0] for i in dom["intents"]]
faltan = [i for i in intents if f"utter_{i}" not in resp and i != "nlu_fallback"]
rotulo(ORIGEN_SINTETICO, "texto de las respuestas del chatbot (código/dominio, no datos de personas)")
print("Respuestas en domain_v3.yml:", len(resp), "| intenciones:", len(intents), "| intenciones sin utter_<intención>:", faltan or "ninguna", "| respuestas vacías:", sin_texto or "ninguna")
print("Respuestas con marcador [Verificar]:", len(con_marca), con_marca)

from openpyxl import load_workbook
wbt = load_workbook(BASE / "docs/tupa/Verificacion_TUPA_v10_4.xlsx", read_only=True, data_only=True)
filas_t = list(wbt["Verificacion"].iter_rows(values_only=True))
resumen_t = {r[0]: (r[1] if len(r) > 1 else None) for r in wbt["Resumen"].iter_rows(values_only=True) if r and isinstance(r[0], str)}
wbt.close()
enc = next(i for i, r in enumerate(filas_t[:6]) if r and r[0] == "#")
col = {str(c): j for j, c in enumerate(filas_t[enc]) if c}
datos_t = [r for r in filas_t[enc + 1:] if r and r[col["Intención"]]]
resultados = pd.Series([str(r[col["Resultado"]]) for r in datos_t]).value_counts().to_dict()
intenciones_tupa = [r[col["Intención"]] for r in datos_t]
con_marca_tupa = [i for i in intenciones_tupa if f"utter_{i}" in con_marca]
rotulo(ORIGEN_REAL, "hoja de verificación del TUPA (valores guardados por Excel); el contenido del TUPA es público, no son datos de personas")
display(pd.DataFrame({"indicador (hoja Resumen)": [k for k in ("Respuestas a verificar", "Pendiente", "Coincide", "Corregir", "No figura en la fuente", "Con resultado (propuesto o confirmado)", "Confirmadas por el tesista", "Prioridad Alta pendientes", "Prioridad Media pendientes", "Filas con alerta", "Corregir o Coincide sin confirmar por el tesista")],
                      "valor": [resumen_t.get(k) for k in ("Respuestas a verificar", "Pendiente", "Coincide", "Corregir", "No figura en la fuente", "Con resultado (propuesto o confirmado)", "Confirmadas por el tesista", "Prioridad Alta pendientes", "Prioridad Media pendientes", "Filas con alerta", "Corregir o Coincide sin confirmar por el tesista")]}))
comparar("G4", "respuestas de trámites a verificar", len(intenciones_tupa), 44, "Nota v11 / hoja Resumen", tol=0)
comparar("G4", "respuestas con [Verificar] entre las 44", len(con_marca_tupa), 0, "Nota v11 («0 de 44»)", tol=0)
comparar("G4", "respuestas con [Verificar] en todo el dominio", len(con_marca), 0, "Nota v11", tol=0)
comparar("G4", "Prioridad Alta pendientes", resumen_t.get("Prioridad Alta pendientes"), 0, "criterio G4", tol=0)
comparar("G4", "filas con alerta", resumen_t.get("Filas con alerta"), 0, "criterio G4", tol=0)
comparar("G4", "Corregir o Coincide sin confirmar", resumen_t.get("Corregir o Coincide sin confirmar por el tesista"), 0, "criterio G4", tol=0)
comparar("G4", "resultado «Corregir» contado en la hoja", resultados.get("Corregir", 0), resumen_t.get("Corregir"), "hoja Resumen", tol=0)
if resumen_t.get("Marcadas [Verificar] en el repositorio") not in (None, 0):
    DECLARADAS.append(f"Verificacion_TUPA_v10_4.xlsx, hoja Resumen: «Marcadas [Verificar] en el repositorio» = {resumen_t['Marcadas [Verificar] en el repositorio']} es un campo manual que NO se actualizó tras aplicar las respuestas; el dominio actual tiene {len(con_marca)} marcadores (comprobado arriba). No se modificó el libro.")
    print("Observación (no se corrige):", DECLARADAS[-1])
mostrar_comparaciones(etapas=["G4"], titulo="Esta sección — recalculado frente a reportado")

▌ORIGEN: Sintético — texto de las respuestas del chatbot (código/dominio, no datos de personas)
Respuestas en domain_v3.yml: 55 | intenciones: 55 | intenciones sin utter_<intención>: ninguna | respuestas vacías: ninguna
Respuestas con marcador [Verificar]: 0 []
▌ORIGEN: Real — hoja de verificación del TUPA (valores guardados por Excel); el contenido del TUPA es público, no son datos de personas


,indicador (hoja Resumen),valor
0,Respuestas a verificar,44
1,Pendiente,9
2,Coincide,0
3,Corregir,28
4,No figura en la fuente,7
5,Con resultado (propuesto o confirmado),35
6,Confirmadas por el tesista,28
7,Prioridad Alta pendientes,0
8,Prioridad Media pendientes,9
9,Filas con alerta,0


Observación (no se corrige): Verificacion_TUPA_v10_4.xlsx, hoja Resumen: «Marcadas [Verificar] en el repositorio» = 40 es un campo manual que NO se actualizó tras aplicar las respuestas; el dominio actual tiene 0 marcadores (comprobado arriba). No se modificó el libro.
Esta sección — recalculado frente a reportado: 7 cifras · coinciden: 7 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,G4,respuestas de trámites a verificar,44,44,Nota v11 / hoja Resumen,Sí
1,G4,respuestas con [Verificar] entre las 44,0,0,Nota v11 («0 de 44»),Sí
2,G4,respuestas con [Verificar] en todo el dominio,0,0,Nota v11,Sí
3,G4,Prioridad Alta pendientes,0,0,criterio G4,Sí
4,G4,filas con alerta,0,0,criterio G4,Sí
5,G4,Corregir o Coincide sin confirmar,0,0,criterio G4,Sí
6,G4,resultado «Corregir» contado en la hoja,28,28,hoja Resumen,Sí


In [8]:
cierre("P09 Diálogo y umbral")

P09 Diálogo y umbral — recalculado frente a reportado: 12 cifras · coinciden: 12 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P09,umbral t congelado para el modelo del lote 2,0.500000,0.500,Protocolo V1.8,Sí
1,P09,ambigüedad congelada,0.100000,0.100,Protocolo V1.8,Sí
2,P09,abstenciones,19.000000,19.000,Protocolo V1.8 / Nota v11,Sí
3,P09,cobertura,0.928839,0.929,"Protocolo V1.8 (92,9 %)",Sí
4,P09,precisión de lo respondido,0.955645,0.956,"Protocolo V1.8 (95,6 %)",Sí
5,G4,respuestas de trámites a verificar,44.000000,44.000,Nota v11 / hoja Resumen,Sí
6,G4,respuestas con [Verificar] entre las 44,0.000000,0.000,Nota v11 («0 de 44»),Sí
7,G4,respuestas con [Verificar] en todo el dominio,0.000000,0.000,Nota v11,Sí
8,G4,Prioridad Alta pendientes,0.000000,0.000,criterio G4,Sí
9,G4,filas con alerta,0.000000,0.000,criterio G4,Sí


DISCREPANCIAS (recalculado ≠ reportado): ninguna
Diferencias CONOCIDAS y ya declaradas (se muestran, no se corrigen):
 - eval_lote2_resumen.json: la sección «umbral» (cobertura 98,9 %, precisión 89,8 %) es un artefacto declarado del script; las cifras oficiales son 92,9 % y 95,6 %.
 - Verificacion_TUPA_v10_4.xlsx, hoja Resumen: «Marcadas [Verificar] en el repositorio» = 40 es un campo manual que NO se actualizó tras aplicar las respuestas; el dominio actual tiene 0 marcadores (comprobado arriba). No se modificó el libro.
Rasa disponible: False · paquete privado: True


## (d) Cómo leer el resultado
- El dominio tiene **55 respuestas** (54 intenciones + `utter_no_entendi`) y **0 con marcador `[Verificar]`**: cada respuesta de trámites pasó por la verificación contra el TUPA (G4: Alta pendientes = 0, alertas = 0, Corregir/Coincide sin confirmar = 0). De las 44 respuestas de trámites, 35 tienen resultado y 28 están confirmadas por el tesista.
- El umbral **abstiene** cuando no está seguro. En el lote 2 eso son **19 abstenciones de 267**: cobertura 92,9 % y, de lo que responde, 95,6 % correcto. Bajar el umbral aumentaría la cobertura a costa de precisión.

## Limitaciones
- El umbral se eligió con datos del lote 1 (71 frases de validación; 185 frases en CV por participante); no se ajustó con el lote 2.
- La cobertura y la precisión dependen del conjunto de 56 situaciones: con situaciones nuevas pueden bajar.
- El texto de las respuestas es información del TUPA; si el TUPA cambia, hay que reaplicar y **re-congelar** (las respuestas forman parte del modelo).